## Structured Outputs
This notebook demonstrates how to work with different output formats and parsers when working with Language Models (LLM).

## What we'll learn:
- Basic string output parsing
- Working with tools and structured outputs
- Using Pydantic models for type validation
- Understanding different parser types and their use cases

In [1]:

import json, os
from dotenv import load_dotenv
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper
# New lines added below to implement parsers for handling different output formats
from typing import Annotated
from pydantic import BaseModel, Field
from lib.parsers import ( #importing the parsers for handling different output formats
    StrOutputParser,
    JsonOutputParser, 
    PydanticOutputParser, 
    ToolOutputParser,
)
# Parser implementation for handling different output formats ends here

load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

True

In [2]:
chat_model = LLM()


## Basic String Output
before diving into more complex output formats, let's understand how to work with simple string outputs from our Language Model.
This demonstrates the most basic form of parsing LLM responses.

In [3]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [4]:
ai_message = chat_model.invoke(messages)
ai_message

AIMessage(content='Event: Science Fair  \nParticipants: Alice and Bob  \nDay: Friday', role='assistant', tool_calls=None)

In [5]:
parser = StrOutputParser()
print(parser.parse(ai_message))

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


## Working with Tools
Now let's see how we can use tools to get structured outputs from our LLM.
Tools help us enforce a specific format for the output and make it easier to process programmatically.

In [6]:
@tool
def calendar_event(name: str, date: str, participants: list[str]):
    """Identify name of the event, date when it will happen and all the participants"""
    return {
        "name": name,
        "date": date,
        "participants": participants
    }

In [7]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [8]:
chat_model_with_tools = LLM(tools=[calendar_event])
ai_message = chat_model_with_tools.invoke(messages)


In [9]:
# New implementation of printing the response, using parsers to handle the structured output   
parser = ToolOutputParser()
structured_output = parser.parse(ai_message)[0]["args"]
print("Structured Output:\n", structured_output)

Structured Output:
 {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


## Using Pydantic Models for Validation
Implement Pydantic models to validate and structure the outputs from the LLM. This step ensures type safety and data integrity.

In [15]:
class CalendarEvent(BaseModel):
    """A Pydantic model representing a calendar event."""
    # Annotated fields with descriptions and default values
    # Annotated[type, metadata]: This is a way to tell Python, 
    # "This variable is a str, but I'm also attaching some extra notes
    # to it." Python itself ignores the notes, but libraries like Pydantic 
    # or FastAPI read them to understand how to handle the data.

    name: Annotated[str, 
                    # Field(...): This is a Pydantic function. 
                    # It’s where you define the "rules" for the variable, like what 
                    # the description is or what happens if the user doesn't provide a value.
                    Field(description="Name/Title of the event. Defaults to ''", default=None)]
    date: Annotated[str, Field(description="Date of the event. Defaults to ''", default=None)]
    participants: Annotated[list[str], Field(description="Who will participate. Defaults to ''", default=None)]


In [11]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [14]:
ai_message = chat_model.invoke(input=messages, response_format=CalendarEvent)
parser = JsonOutputParser()
json_output = parser.parse(ai_message)
print("JSON Output:\n", json.dumps(json_output, indent=4))


JSON Output:
 {
    "name": "Science Fair",
    "date": "Friday",
    "participants": [
        "Alice",
        "Bob"
    ]
}


In [16]:
parser = PydanticOutputParser(model_class=CalendarEvent)
event: CalendarEvent = parser.parse(ai_message)
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [17]:
event.participants

['Alice', 'Bob']